# Equity Duration Notebook

Ziel: Konstruktion mehrerer Equity-Duration-Maße für STOXX Europe 600 Unternehmen, jeweils streng as-of Year-End.

Die Pipeline ist in 7 klaren Schritten organisiert:

1. Setup, Pfade, Session
2. Universum laden (STOXX 600 EURO-HQ)
3. Helper-Funktionen (Pulls + Finance-Math)
4. Daily Prices → Daily Returns (Basisdaten)
5. Firm-level Beta und CAPM Cost of Equity
6. Analyst CFPS Forecasts (FY0–FY4)
7. Year-end Prices, 
8. CFPS Duration-Maße
9. Empirische Duration
10. Final Table für weitere Analyse

Hinweis: CFPSMean_* ist hier operating cash flow per share (CFO/share), nicht Net-Payout.


## Step 1 — Setup, Imports, Pfade, Session

Was hier passiert:
- Bibliotheken importieren
- Projektpfade und I/O-Konventionen setzen
- LSEG-Session vorbereiten (falls nötig)


In [76]:
import re
import numpy as np
import pandas as pd
import lseg.data as ld
from pathlib import Path
import time
import random
import hashlib

import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="lseg")

# Speicherstruktur für Intermediate und Final Output
BASE_DIR = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)

(TABLE_DIR := BASE_DIR / "tables").mkdir(exist_ok=True)
(DATA_DIR := BASE_DIR / "intermediate").mkdir(exist_ok=True)


def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")


def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)

## Step 2 — Universum laden

Was hier passiert:
- Constituents aus der Membership-Matrix laden
- Spalten harmonisieren (z. B. RIC, Name, CountryHQ, MarketCap)
- Ein sauberes Firmen-Universum für alle folgenden Pulls erzeugen


In [70]:
# --- User input (EURO500 instead of STOXX600) ---
# expects: Project_Data/intermediate/euro500.parquet
df_euro500 = load_parquet("euro500")

# Required columns
keep_cols = ["MasterID", "RIC_current", "RIC", "name", "hq_country", "hq_code"]
missing = [c for c in keep_cols if c not in df_euro500.columns]
if missing:
    raise ValueError(
        f"Missing columns in euro500: {missing}. Available: {list(df_euro500.columns)}"
    )

tmp = df_euro500[keep_cols].copy()

# Clean strings (important: empty strings are not NaN)
for c in ["MasterID", "RIC_current", "RIC", "name", "hq_country", "hq_code"]:
    tmp[c] = tmp[c].astype(str).str.strip()
    tmp.loc[tmp[c] == "", c] = pd.NA

# Rename to your standard names
tmp = tmp.rename(
    columns={
        "name": "CompanyName",
        "hq_country": "CountryHQ",
        "hq_code": "CountryCode",
        "RIC": "RIC_snapshot",
    }
)

# Keep only rows with a usable stable id (DO NOT require RIC_current here!)
tmp = tmp.dropna(subset=["MasterID"]).copy()

# Build firm-level table (one row per MasterID)
# - take first non-missing values for metadata
# - keep a representative RIC_current (first non-missing) for convenience/debug


def first_nonmissing(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else pd.NA


df_basic = (
    tmp.groupby("MasterID", as_index=False)
    .agg(
        {
            "CompanyName": first_nonmissing,
            "CountryHQ": first_nonmissing,
            "CountryCode": first_nonmissing,
            "RIC_snapshot": first_nonmissing,
            "RIC_current": first_nonmissing,  # representative; NOT guaranteed unique
        }
    )
    .rename(columns={"RIC_current": "RIC_current_rep"})
    .sort_values("MasterID")
    .reset_index(drop=True)
)

print(
    f"Loaded EURO500 firm table with {df_basic['MasterID'].nunique()} unique firms (MasterID)."
)

# ============================================================
# Build RIC pool for historical data pulls
# ============================================================

ric_pool = pd.concat(
    [
        df_const[["MasterID", "RIC"]].rename(columns={"RIC": "RIC_any"}),
        df_const[["MasterID", "RIC_current"]].rename(
            columns={"RIC_current": "RIC_any"}
        ),
    ],
    ignore_index=True,
)

# Clean identifiers
ric_pool["MasterID"] = ric_pool["MasterID"].astype(str).str.strip()
ric_pool["RIC_any"] = ric_pool["RIC_any"].astype(str).str.strip()

ric_pool.loc[ric_pool["MasterID"] == "", "MasterID"] = pd.NA
ric_pool.loc[ric_pool["RIC_any"] == "", "RIC_any"] = pd.NA

ric_pool = (
    ric_pool.dropna(subset=["MasterID", "RIC_any"])
    .drop_duplicates()
    .reset_index(drop=True)
)

print("RIC pool — unique MasterID:", ric_pool["MasterID"].nunique())
print("RIC pool — unique RIC_any :", ric_pool["RIC_any"].nunique())
print("RIC pool — pairs          :", len(ric_pool))

Loaded EURO500 firm table with 1551 unique firms (MasterID).
RIC pool — unique MasterID: 1551
RIC pool — unique RIC_any : 1722
RIC pool — pairs          : 2153


## Step 3 — Helper-Funktionen: Datenpull und Finance-Math

Was hier passiert:
- Kleine, wiederverwendbare Funktionen definieren
- Typische Bausteine: Year-end Price, Beta, CAPM, Duration-Formeln
- Ziel: Logik kapseln, später weniger Copy/Paste


In [71]:
def coe_capm(beta: float, rf: float, erp: float) -> float:
    """CAPM cost of equity in decimals."""
    if beta is None or (isinstance(beta, float) and np.isnan(beta)):
        return np.nan
    return rf + beta * erp


def duration_discounted(CF: np.ndarray, P: float, r: float) -> float:
    """Implied duration with terminal value inferred from price."""
    if not np.isfinite(P) or P <= 0 or not np.isfinite(r) or r <= 0:
        return np.nan
    CF = np.asarray(CF, dtype=float)
    CF = CF[np.isfinite(CF)]
    T = len(CF)
    if T < 2:
        return np.nan

    t = np.arange(1, T + 1, dtype=float)
    pv_cf = CF / (1.0 + r) ** t
    pv_sum = pv_cf.sum()

    term1 = (t * pv_cf).sum() / P
    term2 = (T + (1.0 + r) / r) * ((P - pv_sum) / P)
    return float(term1 + term2)


def duration_undiscounted(CF: np.ndarray) -> float:
    """Cash-flow timing duration (undiscounted)."""
    CF = np.asarray(CF, dtype=float)
    CF = CF[np.isfinite(CF)]
    T = len(CF)
    if T < 2:
        return np.nan
    t = np.arange(1, T + 1, dtype=float)
    return float((t * CF).sum() / CF.sum()) if CF.sum() != 0 else np.nan

## Step 5 — Firm-level Beta und CAPM Cost of Equity

Was hier passiert:
- Marktserie definieren (z. B. .STOXX)
- Rolling/Window-Betas pro Firma schätzen (5Y, daily)
- Daraus CAPM-Kapitalkosten r_i ableiten

Output (typisch):
- beta_panel (RIC × Year)
- COE_CAPM als Input fürs Discounting


### Hintergrund: CAPM-Beta (kurz)

Definition:
$$
\beta_i = \frac{\mathrm{Cov}(r_{i,t}, r_{m,t})}{\mathrm{Var}(r_{m,t})}
$$

Intuition:
- Beta ist die Sensitivität der Aktie gegenüber dem Markt
- Empirisch: Steigung aus der Regression r_{i,t} auf r_{m,t}
- Hier: Beta wird mit 5-Jahres-Fenstern aus Daily Returns geschätzt


In [ ]:
# ============================================================
# Inputs
# ============================================================

RF = 0.025  # risk-free rate (decimal)
ERP = 0.05  # equity risk premium (decimal)

MKT_RIC = ".STOXX"
SDate = "1999-01-01"
EDate = "2025-12-31"

RI_CLIP = (-0.50, 0.50)
RM_CLIP = (-0.20, 0.20)

MIN_OBS = 400


# ============================================================
# Helper: CAPM cost of equity
# ============================================================
def coe_capm(beta: float, rf: float = RF, erp: float = ERP) -> float:
    if beta is None or (isinstance(beta, float) and np.isnan(beta)):
        return np.nan
    return rf + beta * erp


# ============================================================
# 1) Prepare stock returns from rets_daily  (NEW IDs)
# ============================================================
rets_daily = rets_daily.copy()
rets_daily["date"] = pd.to_datetime(rets_daily["date"], errors="coerce")
rets_daily["ret"] = pd.to_numeric(rets_daily["ret"], errors="coerce")

# Expect new columns
need = ["date", "MasterID", "RIC_current", "ret"]
missing = [c for c in need if c not in rets_daily.columns]
if missing:
    raise ValueError(
        f"rets_daily missing columns: {missing}. Available: {list(rets_daily.columns)}"
    )

rets_daily = rets_daily.dropna(subset=["date", "MasterID", "ret"]).copy()

# If ret looks like percent (e.g. -1.59), convert to decimal; otherwise keep
med_abs = float(np.nanmedian(np.abs(rets_daily["ret"].values)))
if med_abs > 1.0:
    rets_daily["ri"] = rets_daily["ret"] / 100.0
    print("[INFO] Stock returns looked like PERCENT. Converted to decimal /100.")
else:
    rets_daily["ri"] = rets_daily["ret"]
    print("[INFO] Stock returns looked like DECIMAL. Kept as-is.")

rets_daily["ri"] = rets_daily["ri"].clip(RI_CLIP[0], RI_CLIP[1])

# Keep only needed columns
stocks = rets_daily[["date", "MasterID", "RIC_current", "ri"]].copy()


# ============================================================
# 2) Pull market index price close and compute market returns
# ============================================================
ld.open_session()

mkt = ld.get_data(
    universe=[MKT_RIC],
    fields=["TR.PriceClose.Date", "TR.PriceClose"],
    parameters={"SDate": SDate, "EDate": EDate, "FRQ": "D"},
)

ld.close_session()

mkt = mkt.rename(columns=lambda c: c.strip())

if "Date" not in mkt.columns or "Price Close" not in mkt.columns:
    raise ValueError(f"Unexpected market data shape. Columns: {mkt.columns.tolist()}")

mkt["Date"] = pd.to_datetime(mkt["Date"], errors="coerce")
mkt["Price Close"] = pd.to_numeric(mkt["Price Close"], errors="coerce")

mkt = (
    mkt.dropna(subset=["Date", "Price Close"])
    .sort_values("Date")
    .drop_duplicates(subset=["Date"], keep="last")
    .set_index("Date")["Price Close"]
)

rm = mkt.pct_change().rename("rm").dropna()
rm = rm.clip(RM_CLIP[0], RM_CLIP[1])


# ============================================================
# 3) Merge stocks with market returns
# ============================================================
df = (
    stocks.merge(rm, left_on="date", right_index=True, how="inner")
    .dropna(subset=["ri", "rm"])
    .sort_values(["MasterID", "date"])
)

print("Stock returns (ri) describe:\n", df["ri"].describe())
print("\nMarket returns (rm) describe:\n", df["rm"].describe())
print("\nRatio std(ri)/std(rm):", float(df["ri"].std() / df["rm"].std()))


# ============================================================
# 4) Fast 5Y beta at each year-end (as-of 31.12.y) using Cov/Var
# ============================================================
def beta_5y_yearends_fast(df: pd.DataFrame, min_obs: int = 400) -> pd.DataFrame:
    """
    df columns: date, MasterID, ri, rm (decimal returns)
    returns: MasterID | Year | AsOf | BETA_5Y | Nobs
    """
    if df is None or df.empty:
        return pd.DataFrame(columns=["MasterID", "Year", "AsOf", "BETA_5Y", "Nobs"])

    d = df.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d = d.dropna(subset=["date", "MasterID", "ri", "rm"])
    if d.empty:
        return pd.DataFrame(columns=["MasterID", "Year", "AsOf", "BETA_5Y", "Nobs"])

    years = np.sort(d["date"].dt.year.unique())
    out = []

    for y in years:
        asof = pd.Timestamp(year=y, month=12, day=31)
        start = asof - pd.DateOffset(years=5)

        w = d[(d["date"] > start) & (d["date"] <= asof)]
        if w.empty:
            continue

        g = w.groupby("MasterID", sort=False)

        n = g["ri"].count()
        mean_ri = g["ri"].mean()
        mean_rm = g["rm"].mean()

        w = w.join(mean_ri.rename("mean_ri"), on="MasterID")
        w = w.join(mean_rm.rename("mean_rm"), on="MasterID")

        crm = w["rm"] - w["mean_rm"]
        cri = w["ri"] - w["mean_ri"]

        w["prod"] = crm * cri
        w["crm2"] = crm * crm

        cov = w.groupby("MasterID")["prod"].mean()
        var = w.groupby("MasterID")["crm2"].mean()

        beta = cov / var
        beta = beta.where(n >= min_obs)

        panel_y = pd.DataFrame(
            {
                "MasterID": beta.index,
                "Year": y,
                "AsOf": f"{y}-12-31",
                "BETA_5Y": beta.values,
                "Nobs": n.reindex(beta.index).values,
            }
        )

        panel_y = panel_y.dropna(subset=["BETA_5Y"])
        if not panel_y.empty:
            out.append(panel_y)

    if not out:
        return pd.DataFrame(columns=["MasterID", "Year", "AsOf", "BETA_5Y", "Nobs"])

    return pd.concat(out, ignore_index=True)


beta_panel = beta_5y_yearends_fast(df, min_obs=MIN_OBS)

# Add CAPM cost of equity (firm-year)
beta_panel["COE_CAPM"] = RF + beta_panel["BETA_5Y"] * ERP

print(
    "\nBeta distribution:\n",
    beta_panel["BETA_5Y"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]),
)

beta_panel.head()

save_parquet(beta_panel, "beta_capm_panel")

## Step 6 — Analyst CFPS Forecasts (FY0–FY4)

Was hier passiert:
- Konsensschätzungen für Operating CFPS as-of Year-End ziehen
- Horizont: FY0 bis FY4
- Ergebnis als firm-year Panel strukturieren (z. B. cfps_panel)


Die CFPS-Panel-Logik ist strikt as-of 31.12.Y:
- Für jedes Jahr Y werden nur Informationen verwendet, die bis 31.12.Y verfügbar sind
- Die Forecasts bilden Timing und Größenordnung erwarteter Cashflows ab
- Diese Größen sind der zentrale Input für die Duration-Konstruktion


In [ ]:
# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
ASOF_YEARS = list(range(1999, 2026))  # 1999 ... 2025
ASOF_MMDD = "12-31"

HORIZONS = ["FY0", "FY1", "FY2", "FY3", "FY4"]
FIELDS = [f"TR.CFPSMean(period={h})" for h in HORIZONS]

BATCH_SIZE = 100  # konservativ → stabil
rics = df_basic["RIC_current"].dropna().astype(str).unique().tolist()

# Where to checkpoint
CFPS_CKPT_DIR = DATA_DIR / "cfps_checkpoints"
CFPS_CKPT_DIR.mkdir(parents=True, exist_ok=True)

BAD_RICS_LOG = DATA_DIR / "cfps_bad_rics.csv"


def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i : i + n]


# ------------------------------------------------------------
# Helpers: checkpoint I/O
# ------------------------------------------------------------
def _ckpt_path(year: int, batch_ix: int) -> Path:
    return CFPS_CKPT_DIR / f"cfps_{year}_batch_{batch_ix:04d}.parquet"


def _atomic_to_parquet(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_parquet(tmp, index=False)
    tmp.replace(path)


def _append_bad_rics(bad_list: list[str], year: int, reason: str):
    if not bad_list:
        return
    df_bad = pd.DataFrame({"RIC": bad_list, "AsOfYear": year, "reason": reason})
    if BAD_RICS_LOG.exists():
        old = pd.read_csv(BAD_RICS_LOG)
        df_bad = pd.concat([old, df_bad], ignore_index=True)
        df_bad = df_bad.drop_duplicates(subset=["RIC", "AsOfYear", "reason"])
    df_bad.to_csv(BAD_RICS_LOG, index=False)


# ------------------------------------------------------------
# Core: robust get_data with recursive batch-splitting
# ------------------------------------------------------------
def _fetch_cfps_for_batch(
    batch: list[str], asof: str, year: int, max_split_depth: int = 10
) -> tuple[pd.DataFrame, list[str]]:
    """
    Try to fetch CFPSMean forecasts for a batch.
    If request fails (e.g., 400), recursively split batch until it works or identifies bad RIC(s).
    Returns: (df, bad_rics)
    """
    if not batch:
        return pd.DataFrame(), []

    # Parameters: keep minimal; CFPSMean(period=FYx) already encodes horizon.
    # SDate/EDate as "as-of" snapshot works in many LSEG setups.
    params = {"SDate": asof, "EDate": asof}

    try:
        df = ld.get_data(universe=batch, fields=FIELDS, parameters=params)
        if df is None or df.empty:
            return pd.DataFrame(), []
        return df, []
    except Exception as e:
        # If single RIC fails -> flag it as bad
        if len(batch) == 1 or max_split_depth <= 0:
            return pd.DataFrame(), [batch[0]]

        # Split batch and try halves
        mid = len(batch) // 2
        left = batch[:mid]
        right = batch[mid:]

        df_l, bad_l = _fetch_cfps_for_batch(
            left, asof, year, max_split_depth=max_split_depth - 1
        )
        df_r, bad_r = _fetch_cfps_for_batch(
            right, asof, year, max_split_depth=max_split_depth - 1
        )

        # Combine
        out = []
        if df_l is not None and not df_l.empty:
            out.append(df_l)
        if df_r is not None and not df_r.empty:
            out.append(df_r)

        df_out = pd.concat(out, axis=0) if out else pd.DataFrame()
        bad = bad_l + bad_r
        return df_out, bad


def _standardize_cfps_df(df: pd.DataFrame, asof: str, year: int) -> pd.DataFrame:
    """
    Standardize a get_data result into:
    RIC, AsOfYear, AsOfDate, FY0..FY4

    Robust to:
    - MultiIndex columns
    - duplicate column names
    - df[col] returning a DataFrame (2D) instead of Series (1D)
    """
    if df is None or getattr(df, "empty", True):
        return pd.DataFrame(columns=["RIC", "AsOfYear", "AsOfDate"] + HORIZONS)

    df = df.copy()

    # ---- Flatten MultiIndex columns (if any) ----
    # LSEG sometimes returns columns like ('TR.CFPSMean(period=FY0)', 'something')
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            " | ".join([str(x) for x in tup if x is not None]).strip()
            for tup in df.columns
        ]

    # Normalize column names to strings
    df.columns = [str(c).strip() for c in df.columns]

    # ---- Identify instrument column ----
    ric_col = None
    for c in df.columns:
        if c.lower() in {"instrument", "ric"}:
            ric_col = c
            break
    if ric_col is None:
        ric_col = df.columns[0]  # fallback

    # ---- Identify value columns ----
    # Best effort: take the next len(HORIZONS) columns after instrument
    if df.shape[1] >= 1 + len(HORIZONS):
        value_cols = list(df.columns[1 : 1 + len(HORIZONS)])
    else:
        # fallback: search by CFPSMean substring
        cand = [c for c in df.columns if "cfpsmean" in c.lower()]
        value_cols = cand[: len(HORIZONS)]

    out = pd.DataFrame(
        {
            "RIC": df[ric_col].astype(str),
            "AsOfYear": year,
            "AsOfDate": pd.to_datetime(asof),
        }
    )

    def _as_series(x):
        """Ensure we return a 1D Series even if x is a DataFrame (duplicate col names)."""
        if isinstance(x, pd.DataFrame):
            # take first column if duplicate name produced a DataFrame
            return x.iloc[:, 0]
        return x  # Series

    # ---- Assign FY0..FY4 safely ----
    for h, c in zip(HORIZONS, value_cols):
        s = _as_series(df[c])
        out[h] = pd.to_numeric(s, errors="coerce")

    return out


# ------------------------------------------------------------
# Pull CFPS forecasts (all RICs, all years) with checkpoints
# ------------------------------------------------------------
def pull_cfps_panel_all_rics_cached(
    rics: list[str], asof_years: list[int], batch_size: int = 100
) -> pd.DataFrame:
    rics = list(pd.Series(rics).dropna().astype(str).unique())
    rows = []

    ld.open_session()
    try:
        for y in asof_years:
            asof = f"{y}-{ASOF_MMDD}"
            print(f"[INFO] AsOf {asof}")

            for b_ix, batch in enumerate(chunks(rics, batch_size), start=1):
                ck = _ckpt_path(y, b_ix)

                # resume if checkpoint exists
                if ck.exists():
                    try:
                        part = pd.read_parquet(ck)
                        rows.append(part)
                        continue
                    except Exception:
                        # corrupt ckpt -> delete and re-pull
                        try:
                            ck.unlink()
                        except Exception:
                            pass

                # fetch with recursive splitting on failures
                raw_df, bad = _fetch_cfps_for_batch(batch, asof=asof, year=y)

                if bad:
                    _append_bad_rics(
                        bad, year=y, reason="get_data_failed_in_recursive_split"
                    )

                part = _standardize_cfps_df(raw_df, asof=asof, year=y)

                # even if empty, still write checkpoint (so we don't retry forever)
                _atomic_to_parquet(part, ck)

                rows.append(part)

                if b_ix % 10 == 0:
                    print(f"  [INFO] year {y}: batch {b_ix} cached ({len(part)} rows)")

    finally:
        ld.close_session()

    if not rows:
        return pd.DataFrame(columns=["RIC", "AsOfYear", "AsOfDate"] + HORIZONS)

    out = pd.concat(rows, ignore_index=True)
    out["AsOfDate"] = pd.to_datetime(out["AsOfDate"], errors="coerce")
    out = out[out["AsOfDate"].notna()].copy()

    # Deduplicate: last value wins
    out = (
        out.sort_values(["RIC", "AsOfDate"])
        .drop_duplicates(["RIC", "AsOfYear"], keep="last")
        .reset_index(drop=True)
    )

    return out


# ------------------------------------------------------------
# Run
# ------------------------------------------------------------
cfps_panel = pull_cfps_panel_all_rics_cached(rics, ASOF_YEARS, batch_size=BATCH_SIZE)

print(cfps_panel.head())
print("\nRows:", len(cfps_panel))
print("Unique RICs:", cfps_panel["RIC"].nunique())
print(
    "Missing FY0:",
    cfps_panel["FY0"].isna().sum() if "FY0" in cfps_panel.columns else "n/a",
)
print(
    "Missing FY4:",
    cfps_panel["FY4"].isna().sum() if "FY4" in cfps_panel.columns else "n/a",
)

save_parquet(cfps_panel, "cfps_forecasts")
print(f"[INFO] Bad RICs log (if any): {BAD_RICS_LOG}")

In [ ]:
# --- Load Cashflows (Parquet) ---
CFPS_FILE = DATA_DIR / "cfps_forecasts.parquet"
cfps_panel = pd.read_parquet(CFPS_FILE)

## Step 7 — Year-end Prices

Was hier passiert (gesamter Output-Block):
- Year-end Preise (31.12.Y) ziehen und mit Forecasts mergen


In [ ]:
# ============================================================
# Pull year-end prices (Price Close) for ALL RICs at each year-end (31-12)
# Output: price_panel with columns: RIC | AsOfYear | AsOfDate | Price_12_31
# ============================================================

ASOF_YEARS = list(range(1999, 2026))  # 1999 ... 2025
ASOF_MMDD = "12-31"
BATCH_SIZE = 200

rics = df_basic["RIC_current"].dropna().astype(str).unique().tolist()


def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i : i + n]


def _find_price_col(df: pd.DataFrame) -> str | None:
    # Your environment typically names it "Price Close"
    if "Price Close" in df.columns:
        return "Price Close"
    # Fallbacks if LSEG changes labeling
    for c in df.columns:
        if "price close" in str(c).lower():
            return c
    # Rare case: direct TR.PriceClose column name
    if "TR.PriceClose" in df.columns:
        return "TR.PriceClose"
    return None


rows = []

ld.open_session()
try:
    for y in ASOF_YEARS:
        asof = f"{y}-{ASOF_MMDD}"
        print(f"Pulling PriceClose for {asof} ...")

        for batch in chunks(rics, BATCH_SIZE):
            df = ld.get_data(
                universe=batch,
                fields=["TR.PriceClose.Date", "TR.PriceClose"],
                parameters={"SDate": asof, "EDate": asof, "FRQ": "D"},
            )

            if df is None or df.empty:
                continue

            df = df.rename(columns=lambda c: str(c).strip())

            # Identifier
            id_col = "Instrument" if "Instrument" in df.columns else df.columns[0]

            # Date column (should exist because we requested TR.PriceClose.Date)
            if "Date" not in df.columns:
                # If your environment returns a different label, try to find it
                date_col = next(
                    (c for c in df.columns if str(c).lower() == "date"), None
                )
                if date_col is None:
                    continue
            else:
                date_col = "Date"

            # Price column
            price_col = _find_price_col(df)
            if price_col is None:
                continue

            tmp = df[[id_col, date_col, price_col]].copy()
            tmp = tmp.rename(
                columns={id_col: "RIC", date_col: "Date", price_col: "Price_12_31"}
            )
            tmp["RIC"] = tmp["RIC"].astype(str)
            tmp["Date"] = pd.to_datetime(tmp["Date"], errors="coerce")
            tmp["Price_12_31"] = pd.to_numeric(tmp["Price_12_31"], errors="coerce")
            tmp = tmp.dropna(subset=["RIC", "Price_12_31"])

            if tmp.empty:
                continue

            tmp["AsOfYear"] = y
            tmp["AsOfDate"] = pd.Timestamp(asof)

            # If duplicates appear (rare), keep last
            tmp = tmp.sort_values(["RIC", "Date"]).drop_duplicates(
                subset=["RIC"], keep="last"
            )

            rows.append(tmp[["RIC", "AsOfYear", "AsOfDate", "Price_12_31"]])

finally:
    ld.close_session()

price_panel = (
    pd.concat(rows, ignore_index=True)
    if rows
    else pd.DataFrame(columns=["RIC", "AsOfYear", "AsOfDate", "Price_12_31"])
)

# Final dedup (last wins)
if not price_panel.empty:
    price_panel = (
        price_panel.sort_values(["RIC", "AsOfDate"])
        .drop_duplicates(["RIC", "AsOfYear"], keep="last")
        .reset_index(drop=True)
    )

print(price_panel.head())
print("Rows:", len(price_panel))
print("Unique RICs:", price_panel["RIC"].nunique())
print("Missing prices:", price_panel["Price_12_31"].isna().sum())

# save
save_parquet(price_panel, "price_year_end_panel")

In [ ]:
# --- Load Cashflows (Parquet) ---
Price_FILE = DATA_DIR / "price_yearend_panel_1999_2025.parquet"
price_panel = pd.read_parquet(Price_FILE)

## Step 8 — Duration-Maße aus CFPS

Was hier passiert:
- Drei Timing-/Duration-Maße berechnen:
  - Duration_undiscounted
  - Duration_r125 (fixed r = 12.5%)
  - Duration_CAPM (firm-spezifisches r_i)
- Alle Maße sind konsistent auf per-share Größen aufgebaut


In [ ]:
HORIZONS = ["FY0", "FY1", "FY2", "FY3", "FY4"]

# --- 0) Ensure all horizon columns exist (optional safety) ---
for c in HORIZONS:
    if c not in cfps_panel.columns:
        cfps_panel[c] = pd.NA

# --- 1) Make horizons numeric and replace pd.NA safely ---
# convert to numeric -> NaN, then enforce float dtype
for c in HORIZONS:
    cfps_panel[c] = pd.to_numeric(cfps_panel[c], errors="coerce").astype(float)

# Build CF matrix (N x 5) as float, guaranteed NaN for missing
CF = cfps_panel[HORIZONS].to_numpy(dtype=float)

# Helper: row-wise apply without pd.NA issues


def _dur_undiscounted_from_row(cf_row: np.ndarray) -> float:
    if np.all(np.isnan(cf_row)):
        return np.nan
    return duration_undiscounted(cf_row)


cfps_panel["Duration_undiscounted"] = np.apply_along_axis(
    _dur_undiscounted_from_row, 1, CF
)

# --- 2) Merge year-end prices (needed as terminal value P) ---
cfps_panel = cfps_panel.merge(
    price_panel[["RIC", "AsOfYear", "Price_12_31"]], on=["RIC", "AsOfYear"], how="left"
)

# --- 3) Discounted duration at fixed r = 12.5% ---
R_FIXED = 0.125
P = pd.to_numeric(cfps_panel["Price_12_31"], errors="coerce").to_numpy(dtype=float)


def _dur_rfixed_from_row(args) -> float:
    cf_row, p = args
    if np.all(np.isnan(cf_row)) or (not np.isfinite(p)) or p <= 0:
        return np.nan
    return duration_discounted(cf_row, P=float(p), r=R_FIXED)


cfps_panel["Duration_r125"] = [
    _dur_rfixed_from_row((cf_row, p)) for cf_row, p in zip(CF, P)
]

cfps_panel[
    ["RIC", "AsOfYear", "Price_12_31", "Duration_undiscounted", "Duration_r125"]
].head()

In [ ]:
HORIZONS = ["FY0", "FY1", "FY2", "FY3", "FY4"]


def row_cf_array(row) -> np.ndarray:
    return row[HORIZONS].to_numpy(dtype=float)


# 1) Bring in firm-year CAPM discount rates r_{i,y}
# beta_panel might use "Year" instead of "AsOfYear"
coe = beta_panel.copy()
if "AsOfYear" not in coe.columns and "Year" in coe.columns:
    coe = coe.rename(columns={"Year": "AsOfYear"})

coe = coe[["RIC", "AsOfYear", "COE_CAPM"]].drop_duplicates(
    ["RIC", "AsOfYear"], keep="last"
)
coe["AsOfYear"] = pd.to_numeric(coe["AsOfYear"], errors="coerce").astype("Int64")
coe["COE_CAPM"] = pd.to_numeric(coe["COE_CAPM"], errors="coerce")

cfps_panel["AsOfYear"] = pd.to_numeric(cfps_panel["AsOfYear"], errors="coerce").astype(
    "Int64"
)

cfps_panel = cfps_panel.merge(coe, on=["RIC", "AsOfYear"], how="left")

# 2) Compute CAPM-discounted duration


def safe_duration_capm(row) -> float:
    cf = row_cf_array(row)
    P = row.get("Price_12_31", np.nan)
    r = row.get("COE_CAPM", np.nan)

    if np.all(np.isnan(cf)) or pd.isna(P) or P <= 0 or pd.isna(r):
        return np.nan

    # Optional guardrail: avoid pathological r values
    if r <= -0.50 or r >= 1.00:
        return np.nan

    return duration_discounted(cf, P=P, r=float(r))


cfps_panel["Duration_CAPM"] = cfps_panel.apply(safe_duration_capm, axis=1)

cfps_panel[
    ["RIC", "AsOfYear", "Price_12_31", "COE_CAPM", "Duration_r125", "Duration_CAPM"]
].head()

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Robust diagnostics for CAPM-DCF Duration (works as-is)
# - Does NOT rely on prior variables like `d`
# - Handles COE_CAPM in % vs decimal
# - Computes PV(explicit), terminal residual/share, FY0 sensitivity
# - Checks how much Duration_CAPM is driven by r
# ============================================================

H0 = ["FY0", "FY1", "FY2", "FY3", "FY4"]
H1 = ["FY1", "FY2", "FY3", "FY4"]


def _to_float_array(row, cols):
    return row[cols].to_numpy(dtype=float)


def pv_explicit(cf, r, t_start=1):
    """PV of explicit cashflows assuming cf[j] arrives at t = t_start + j."""
    t = np.arange(t_start, t_start + len(cf), dtype=float)
    return np.nansum(cf / ((1.0 + r) ** t))


def build_diag_df(cfps_panel: pd.DataFrame) -> pd.DataFrame:
    d = cfps_panel.copy()

    # Ensure required cols exist
    needed = ["RIC", "AsOfYear", "Price_12_31", "COE_CAPM", "Duration_CAPM"] + H0
    missing = [c for c in needed if c not in d.columns]
    if missing:
        raise KeyError(f"Missing required columns in cfps_panel: {missing}")

    # numeric coercions
    d["Price_12_31"] = pd.to_numeric(d["Price_12_31"], errors="coerce")
    d["COE_CAPM"] = pd.to_numeric(d["COE_CAPM"], errors="coerce")
    d["Duration_CAPM"] = pd.to_numeric(d["Duration_CAPM"], errors="coerce")

    # Keep rows where diagnostics are possible
    keep = (
        d["Price_12_31"].notna()
        & (d["Price_12_31"] > 0)
        & d["COE_CAPM"].notna()
        & (~d[H0].isna().all(axis=1))
    )
    d = d.loc[keep].copy()

    # --- Heuristic: detect whether COE_CAPM is in percent or decimal
    # If median > 1, it's almost surely percent (e.g., 7.5 instead of 0.075)
    assume_percent = d["COE_CAPM"].median() > 1
    d["r_dec"] = d["COE_CAPM"] / 100.0 if assume_percent else d["COE_CAPM"]

    # Clamp to plausible range to avoid explosive PVs (adjust if you want)
    d["r_dec_clamped"] = d["r_dec"].clip(lower=0.00, upper=0.40)

    # --- PV + terminal diagnostics
    # Version A: FY0..FY4 treated as years 1..5 (t_start=1)
    d["PV_exp_FY0_t1"] = d.apply(
        lambda row: pv_explicit(
            _to_float_array(row, H0), row["r_dec_clamped"], t_start=1
        ),
        axis=1,
    )
    d["Terminal_resid_FY0_t1"] = d["Price_12_31"] - d["PV_exp_FY0_t1"]
    d["Terminal_share_FY0_t1"] = d["Terminal_resid_FY0_t1"] / d["Price_12_31"]

    # Version B: drop FY0 (FY1..FY4 treated as years 1..4)
    d["PV_exp_FY1_t1"] = d.apply(
        lambda row: pv_explicit(
            _to_float_array(row, H1), row["r_dec_clamped"], t_start=1
        ),
        axis=1,
    )
    d["Terminal_resid_FY1_t1"] = d["Price_12_31"] - d["PV_exp_FY1_t1"]
    d["Terminal_share_FY1_t1"] = d["Terminal_resid_FY1_t1"] / d["Price_12_31"]

    d["Delta_terminal_share_FY0minusFY1"] = (
        d["Terminal_share_FY0_t1"] - d["Terminal_share_FY1_t1"]
    )

    # Flags
    d["FLAG_r_weird"] = (d["r_dec"] < 0.00) | (d["r_dec"] > 0.40)
    d["FLAG_term_negative_FY0"] = d["Terminal_resid_FY0_t1"] <= 0
    d["FLAG_term_dominate_FY0"] = d["Terminal_share_FY0_t1"] >= 0.70

    # store the heuristic decision (nice to print later)
    d.attrs["assume_percent"] = assume_percent
    return d


# ---------- Run diagnostics ----------
diag = build_diag_df(cfps_panel)

print("\n=== (1) COE_CAPM scaling check ===")
print("Assume COE_CAPM is percent?", diag.attrs.get("assume_percent"))
print("COE_CAPM raw summary:")
print(
    diag["COE_CAPM"].describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
    )
)
print("\nr_dec (interpreted decimal) summary:")
print(
    diag["r_dec"].describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\n=== (2) Terminal diagnostics ===")
for tag in ["FY0_t1", "FY1_t1"]:
    neg_term = (diag[f"Terminal_resid_{tag}"] <= 0).mean()
    big_term = (diag[f"Terminal_share_{tag}"] >= 0.70).mean()
    print(f"\n--- {tag} ---")
    print(f"Share(terminal residual <= 0): {neg_term:.3%}")
    print(f"Share(terminal share >= 70%): {big_term:.3%}")
    print("Terminal share summary:")
    print(
        diag[f"Terminal_share_{tag}"].describe(
            percentiles=[0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
        )
    )

print("\n=== (3) FY0 sensitivity (terminal share difference) ===")
print(
    diag["Delta_terminal_share_FY0minusFY1"].describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\n=== (4) Duration_CAPM sanity + dependence on r ===")
dur = diag[
    [
        "Duration_CAPM",
        "r_dec",
        "r_dec_clamped",
        "Price_12_31",
        "Terminal_share_FY0_t1",
        "Terminal_share_FY1_t1",
    ]
].dropna()
print("\nDuration_CAPM summary:")
print(
    dur["Duration_CAPM"].describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nPearson correlations with Duration_CAPM:")
print(dur.corr(numeric_only=True)["Duration_CAPM"].sort_values())

print("\n=== (5) Flags ===")
flag_shares = pd.Series(
    {
        "r_weird": diag["FLAG_r_weird"].mean(),
        "terminal_negative_FY0": diag["FLAG_term_negative_FY0"].mean(),
        "terminal_dominate_FY0(>=70%)": diag["FLAG_term_dominate_FY0"].mean(),
    }
).map(lambda x: f"{x:.3%}")
print(flag_shares)

# ---------- Top / Bottom 10 (no merges, no KeyErrors) ----------
display_cols = [
    "RIC",
    "AsOfYear",
    "Duration_CAPM",
    "r_dec",
    "Price_12_31",
    "Terminal_share_FY0_t1",
    "Terminal_share_FY1_t1",
] + H0

top10 = (
    diag.loc[diag["Duration_CAPM"].notna(), display_cols]
    .sort_values("Duration_CAPM", ascending=False)
    .head(10)
)
bot10 = (
    diag.loc[diag["Duration_CAPM"].notna(), display_cols]
    .sort_values("Duration_CAPM", ascending=True)
    .head(10)
)

print("\nTop 10 Duration_CAPM rows:")
display(top10)

print("\nBottom 10 Duration_CAPM rows:")
display(bot10)

# Optional: dataset of problematic rows
problem_rows = diag.loc[
    diag["FLAG_r_weird"]
    | diag["FLAG_term_negative_FY0"]
    | diag["FLAG_term_dominate_FY0"],
    [
        "RIC",
        "AsOfYear",
        "Price_12_31",
        "COE_CAPM",
        "r_dec",
        "r_dec_clamped",
        "PV_exp_FY0_t1",
        "Terminal_resid_FY0_t1",
        "Terminal_share_FY0_t1",
    ]
    + H0,
].copy()

print(f"\nProblem rows: {len(problem_rows):,} / {len(diag):,}")
# display(problem_rows.head(20))

In [ ]:
import numpy as np
import pandas as pd


def duration_discounted_robust(cf, P, r, t_start=1, allow_negative_cf=True):
    """
    Robust Macaulay-style equity duration with terminal as residual (P - PV_explicit),
    but TERMINAL IS FLOORED at 0 to avoid negative-terminal explosions.

    cf: array-like of forecast cashflows (e.g., FY1..FY4), in same units as P
    P : price (same scale as PV of cf)
    r : discount rate (decimal)
    t_start: first cashflow timing (1 means first element is at t=1)
    allow_negative_cf: if False, negative forecast CFs are set to 0
    """
    cf = np.array(cf, dtype=float)

    if not allow_negative_cf:
        cf = np.where(np.isnan(cf), np.nan, np.maximum(cf, 0.0))

    if (
        np.all(np.isnan(cf))
        or (P is None)
        or (not np.isfinite(P))
        or P <= 0
        or (not np.isfinite(r))
        or r <= -0.5
    ):
        return np.nan

    # times
    t = np.arange(t_start, t_start + len(cf), dtype=float)

    # PV explicit
    disc = (1.0 + r) ** t
    pv_exp = np.nansum(cf / disc)

    # terminal residual (floor at 0)
    terminal = P - pv_exp
    if not np.isfinite(terminal):
        return np.nan
    terminal = max(0.0, terminal)

    # weights (PV shares)
    pv_cf = np.where(np.isfinite(cf), cf / disc, 0.0)
    pv_total = pv_exp + terminal

    # if pv_total is 0 (pathological), return nan
    if pv_total <= 0:
        return np.nan

    w_cf = pv_cf / pv_total
    # terminal "maturity": Macaulay duration of perpetuity starting at T (approx)
    # if terminal is 0, terminal contribution is 0 anyway
    T = t[-1]  # last explicit time
    D_terminal = T + (1.0 + r) / r if (r > 0 and terminal > 0) else 0.0
    w_terminal = terminal / pv_total

    # duration
    D = np.nansum(t * w_cf) + w_terminal * D_terminal
    return float(D)


# --- Apply (recommended: drop FY0) ---
H_use = ["FY1", "FY2", "FY3", "FY4"]


def safe_duration_capm_robust(row):
    P = row.get("Price_12_31", np.nan)
    r = row.get("COE_CAPM", np.nan)
    cf = row[H_use].to_numpy(dtype=float)

    if pd.isna(P) or P <= 0 or pd.isna(r):
        return np.nan
    if r <= 0.0 or r >= 0.50:
        return np.nan

    return duration_discounted_robust(
        cf, P=P, r=float(r), t_start=1, allow_negative_cf=True
    )


cfps_panel["Duration_CAPM_fix"] = cfps_panel.apply(safe_duration_capm_robust, axis=1)

print(
    cfps_panel["Duration_CAPM_fix"].describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
    )
)

In [ ]:
# 1) Extremwerte check
print(
    cfps_panel["Duration_CAPM_fix"].describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
    )
)

# 2) Wie viele NaNs?
print("NaN share:", cfps_panel["Duration_CAPM_fix"].isna().mean())

## Step 9 — Empirische Duration: Zins-Sensitivität

Was hier passiert:
- Daily Delta-y (2Y EUR OIS) mit Daily Returns mergen
- Pro Firma eine einfache Sensitivität schätzen:
$$
r_{i,t} = a_i + b_i \Delta y_t + \varepsilon_{i,t}
\quad\Rightarrow\quad
D^{emp}_i = -b_i
$$
- Ergebnis: eine datengetriebene Duration-Approximation

### 2Y OIS Serie ziehen (RIC EUREON2Y=)

Wir verwenden das verfügbare Feld, z. B. TR.MIDPRICE.


In [ ]:
ld.open_session()

# 2Y EUR OIS / EONIA-style series
ric_rate = "EUREON2Y="
field_rate = "TR.MIDPRICE"

y_df = ld.get_history(
    universe=[ric_rate],
    fields=[field_rate],
    start="1999-01-01",
    end="2025-12-31",
    interval="daily",
)

ld.close_session()

# Build daily rate changes (in percentage points)
rate_df = y_df.copy().reset_index()
rate_df.columns = ["date", "y"]  # y in percent
rate_df["date"] = pd.to_datetime(rate_df["date"])
rate_df["y"] = pd.to_numeric(rate_df["y"], errors="coerce")
rate_df = rate_df.dropna(subset=["y"]).sort_values("date")
rate_df["dy"] = rate_df["y"].diff()

rates_daily = rate_df[["date", "dy"]].dropna()

rates_daily.head()

In [ ]:
df_2yOIS = rates_daily.copy()
save_parquet(df_2yOIS, "rates_2yOIS_daily")

In [ ]:
# --- Load Cashflows (Parquet) ---
df_2yOIS_FILE = DATA_DIR / "rates_2yOIS_daily.parquet"
rates_daily = pd.read_parquet(df_2yOIS_FILE)

### Firm-level Rate Betas und empirische Duration schätzen


In [ ]:
def estimate_rate_beta_duration(
    rets_daily: pd.DataFrame,
    rates_daily: pd.DataFrame,
    min_obs: int = 200,
    agg_ret: str = "last",  # how to resolve duplicate (date,RIC) returns
) -> pd.DataFrame:
    """
    Estimate: r_{i,t} = a_i + b_i * dy_t
    Returns:
      - beta_rate_2yOIS = b_i
      - D_emp_2yOIS     = -b_i
      - n_obs           = number of daily observations used

    Notes:
    - Handles duplicate (date, RIC) rows in rets_daily by aggregating.
    - Keeps only dates where both stock returns and dy are observed.
    - Does NOT change units: make sure ret and dy are in compatible units for interpretation.
      (If ret is in percent and dy is in percentage points, b is "percent per pp".)
    """

    # --- 0) Defensive copies + types ---
    r = rets_daily.copy()
    z = rates_daily.copy()

    r["date"] = pd.to_datetime(r["date"], errors="coerce")
    z["date"] = pd.to_datetime(z["date"], errors="coerce")

    r["RIC"] = r["RIC"].astype(str)
    r["ret"] = pd.to_numeric(r["ret"], errors="coerce")
    z["dy"] = pd.to_numeric(z["dy"], errors="coerce")

    r = r.dropna(subset=["date", "RIC", "ret"])
    z = z.dropna(subset=["date", "dy"])

    # --- 1) Resolve duplicates: one return per (date, RIC) ---
    # pivot() fails if duplicates exist; we handle it explicitly here.
    if agg_ret == "last":
        r = r.sort_values(["RIC", "date"]).drop_duplicates(
            subset=["RIC", "date"], keep="last"
        )
    elif agg_ret == "mean":
        r = r.groupby(["RIC", "date"], as_index=False)["ret"].mean()
    else:
        raise ValueError("agg_ret must be 'last' or 'mean'.")

    # --- 2) Merge with rates and drop missing ---
    df = (
        r.merge(z[["date", "dy"]], on="date", how="inner")
        .dropna(subset=["ret", "dy"])
        .copy()
    )

    # --- 3) OLS per RIC: y = a + b x ---
    out = []
    for ric, g in df.groupby("RIC", sort=False):
        if len(g) < min_obs:
            continue

        x = g["dy"].to_numpy(dtype=float)
        y = g["ret"].to_numpy(dtype=float)

        # Drop any remaining non-finite entries
        m = np.isfinite(x) & np.isfinite(y)
        x = x[m]
        y = y[m]
        if len(y) < min_obs:
            continue

        X = np.column_stack([np.ones_like(x), x])
        alpha, b = np.linalg.lstsq(X, y, rcond=None)[0]

        out.append(
            {
                "RIC": ric,
                "beta_rate_2yOIS": float(b),
                "D_emp_2yOIS": float(-b),
                "n_obs": int(len(y)),
            }
        )

    return pd.DataFrame(out)


# -----------------------------
# Run + merge into df_basic
# -----------------------------
dur_emp = estimate_rate_beta_duration(
    rets_daily, rates_daily, min_obs=200, agg_ret="last"
)

df_basic = df_basic.merge(dur_emp, on="RIC", how="left")

print(df_basic[["RIC", "D_emp_2yOIS", "beta_rate_2yOIS", "n_obs"]].head())
print(df_basic["D_emp_2yOIS"].describe())

# Spearman correlation (requires both columns present)
cols = [c for c in ["Duration_r125", "D_emp_2yOIS"] if c in df_basic.columns]
if len(cols) == 2:
    print(df_basic[cols].corr(method="spearman"))
else:
    print("Duration_r125 not found in df_basic (compute/merge it first).")

print("Share with D_emp:", df_basic["D_emp_2yOIS"].notna().mean())

## Step 11 — Final Results Table (gerundet und exportierbar)

Was hier passiert:
- Relevante firm-year Outputs zusammenführen
- Spalten benennen und aufräumen
- Ergebnis als finale Analyse-/Export-Tabelle speichern

In [ ]:
# ============================================================
# Firm-year panel: durations + empirical duration + 5Y beta
# ============================================================

# 1) Base: time-varying durations from cfps_panel
dur_cols = [
    c
    for c in ["Duration_undiscounted", "Duration_r125", "Duration_CAPM"]
    if c in cfps_panel.columns
]
firm_year = cfps_panel[["RIC", "AsOfYear"] + dur_cols].copy()
firm_year = firm_year.rename(columns={"AsOfYear": "YEAR"})
firm_year["YEAR"] = pd.to_numeric(firm_year["YEAR"], errors="coerce").astype("Int64")

# 2) Add time-invariant empirical duration (rename)
emp = (
    dur_emp[["RIC", "D_emp_2yOIS"]].copy()
    if ("dur_emp" in globals() and dur_emp is not None and not dur_emp.empty)
    else df_basic[["RIC", "D_emp_2yOIS"]].drop_duplicates("RIC").copy()
)
emp = emp.rename(columns={"D_emp_2yOIS": "Duration_emp_2yOIS"})
firm_year = firm_year.merge(emp, on="RIC", how="left")

# 3) Add 5Y beta from beta_panel (RIC x Year)
beta_year = (
    beta_panel[["RIC", "Year", "BETA_5Y"]].copy().rename(columns={"Year": "YEAR"})
)
beta_year["YEAR"] = pd.to_numeric(beta_year["YEAR"], errors="coerce").astype("Int64")
beta_year = (
    beta_year.sort_values(["RIC", "YEAR"])
    .dropna(subset=["BETA_5Y"])
    .drop_duplicates(["RIC", "YEAR"], keep="last")
)

# rerun-safe merge
firm_year = firm_year.drop(columns=["BETA_5Y"], errors="ignore").merge(
    beta_year, on=["RIC", "YEAR"], how="left"
)
firm_year = firm_year.sort_values(["RIC", "YEAR"]).reset_index(drop=True)

save_parquet(firm_year, "durations_panel")
firm_year.head()